# 06 — Draft, bans and hero meta

The draft layer: what the meta values (contest rate), what teams ban and
pick, how wide each team's and player's hero pool is, and — critically —
how the meta **shifted across the last patch**, which is what invalidates
pre-patch form. Also produces the ban table that prices the *Most Banned
Hero* market in notebook 08.

In [ ]:
import sys, pathlib
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == 'notebooks' else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT))
import pandas as pd, numpy as np
pd.set_option('display.max_columns', 60); pd.set_option('display.width', 160)
DATA = ROOT / 'data'


In [ ]:
from src.draft import (ban_table, contest_table, hero_winrates,
                       team_draft_style, player_hero_pool, meta_shift)
matches = pd.read_parquet(DATA / 'matches.parquet')
pb      = pd.read_parquet(DATA / 'picks_bans.parquet')
mp      = pd.read_parquet(DATA / 'match_players.parquet')
teams   = pd.read_parquet(DATA / 'teams.parquet')[['team_id','name']]
heroes  = pd.read_parquet(DATA / 'heroes.parquet') if (DATA/'heroes.parquet').exists() else None
print(f'{pb.match_id.nunique():,} drafted games, {len(pb):,} draft actions')
bans = ban_table(pb, heroes, matches); bans.head(20)

In [ ]:
# Contest rate is the honest meta measure: picks+bans per game.
# A hero at ~1.0 is removed from the board every single game.
ct = contest_table(pb, heroes)
ct.head(25)

In [ ]:
# Which heroes actually WIN when they get through the draft?
hw = hero_winrates(pb, matches, heroes, min_picks=10)
print('Best:'); print(hw.head(12).to_string(index=False))
print('\nWorst:'); print(hw.tail(12).to_string(index=False))

In [ ]:
# Team draft signatures: wide/unpredictable vs narrow/comfort-based
tds = team_draft_style(pb, matches, teams)
tds.head(16)

In [ ]:
# Player hero pools (breadth + signature heroes)
player_hero_pool(mp, heroes, min_games=10).head(25)

In [ ]:
# META SHIFT across the last patch boundary — run this the day a patch lands.
patches = sorted(matches.patch.dropna().unique())
print('patches present:', patches)
if len(patches) >= 2:
    shift = meta_shift(pb, matches, int(patches[-2]), int(patches[-1]), heroes)
    display(shift)
else:
    print('need two patches in the sample to compute a shift')